# Laboratorio 7 - Spark MLlib
### CC3066 Data Science, UVG, Semestre II 2026

Carga y armonización de las bases de Personas de la ENEIC (Ejercicio 1) y correlaciones entre variables numéricas (Ejercicio 3).

Nota: la sección 3 lee `prep_2025`, que se genera en la parte de filtros del Ejercicio 1.

## 0. Configuración

Todas las rutas y constantes se definen aquí una sola vez. `BASE_DIR` apunta al volumen montado por `docker-compose` (`./working_dir` en `/opt/app/working_dir`); se puede sobrescribir con la variable de entorno `LAB7_BASE` si se trabaja fuera del contenedor.

In [ ]:
import os
import gc
from functools import reduce
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from pyspark.sql import SparkSession, DataFrame
from pyspark.sql import functions as F
from pyspark.sql import types as T
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.stat import Correlation

SEED = 42
MAX_FILAS_GRAFICO = 5_000   # tope de filas que se transfieren a pandas para dibujar

BASE_DIR      = Path(os.environ.get("LAB7_BASE", "/opt/app/working_dir"))
RAW_DIR       = BASE_DIR / "raw"
PARQUET_DIR   = BASE_DIR / "parquet"
STAGING_DIR   = PARQUET_DIR / "staging"
PREP_2025_DIR = PARQUET_DIR / "prep_2025"
PREP_2026_DIR = PARQUET_DIR / "prep_2026"
MODELS_DIR    = BASE_DIR / "models"
FIG_DIR       = BASE_DIR / "figures"

for d in (RAW_DIR, STAGING_DIR, MODELS_DIR, FIG_DIR):
    d.mkdir(parents=True, exist_ok=True)

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)
sns.set_theme(style="whitegrid")

In [ ]:
spark = (
    SparkSession.builder
    .appName("lab7-spark-mllib")
    .master("local[*]")
    .config("spark.driver.memory", "4g")
    .config("spark.sql.shuffle.partitions", "8")
    # Arrow desactivado: con algunas versiones de Java falla en toPandas(); los volúmenes son pequeños
    .config("spark.sql.execution.arrow.pyspark.enabled", "false")
    .config("spark.sql.session.timeZone", "UTC")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")
print("Spark:", spark.version)
assert spark.version.startswith("3.5"), "El laboratorio requiere Spark 3.5.x"

### 0.1 Catálogo de archivos

El período se asigna a partir del archivo de procedencia, no de la columna `TRIMESTRE`. Si los nombres de los Excel en `working_dir/raw/` son distintos, basta con cambiar la columna `archivo` de esta tabla.

`registros_esperados` y `columnas_esperadas` son los conteos publicados en el enunciado; se usan para verificar que se leyó el archivo correcto y completo.

In [ ]:
ARCHIVOS = pd.DataFrame([
    # periodo_archivo, anio_archivo, trimestre_calendario, archivo, uso, registros_esperados, columnas_esperadas
    ("2025T1", 2025, 1, "personas_2025T1.xlsx", "train",      51_588, 270),
    ("2025T2", 2025, 2, "personas_2025T2.xlsx", "train",      51_167, 270),
    ("2025T3", 2025, 3, "personas_2025T3.xlsx", "train",      51_583, 270),
    ("2025T4", 2025, 4, "personas_2025T4.xlsx", "validacion", 49_338, 302),
    ("2026T1", 2026, 1, "personas_2026T1.xlsx", "test",       49_843, 270),
], columns=["periodo_archivo", "anio_archivo", "trimestre_calendario", "archivo",
            "uso", "registros_esperados", "columnas_esperadas"])

ARCHIVOS["existe"] = ARCHIVOS["archivo"].map(lambda a: (RAW_DIR / a).exists())
display(ARCHIVOS)

faltan = ARCHIVOS.loc[~ARCHIVOS["existe"], "archivo"].tolist()
assert not faltan, f"No se encontraron en {RAW_DIR}: {faltan}"

### 0.2 Columnas seleccionadas y tipo destino

Cada columna original se renombra a su nombre analítico y se le asigna una clase de tipo:

| Clase | Tipo Spark | Tratamiento |
|---|---|---|
| `num` | `DoubleType` | Cantidades continuas o conteos (salario, edad, antigüedad, horas, factor). Se dejan en `double` para poder verificar después que sean finitas y que los meses sean enteros. |
| `cat` | `StringType` | Códigos categóricos normalizados como texto (`"01"`, `1`, `1.0` → `"1"`). Se conserva como texto para no perder valores no numéricos: la validación contra el diccionario y la asignación de `DESCONOCIDO` se hacen en la etapa de filtros. |
| `int` | `IntegerType` | Códigos usados como filtro o auditoría (`OCUPADOS`, `ANIO`, `TRIMESTRE`). |
| `id` | `LongType` | Identificadores de auditoría (`NUM_HOGAR`, `NUM_PERSONA`). |

In [ ]:
COLUMNAS = {
    # original      (nombre analítico,         clase)
    "P05D01":      ("salario_mensual",        "num"),
    "P02A03":      ("edad",                   "num"),
    "P05C07A":     ("antiguedad_anios",       "num"),
    "P05C07B":     ("antiguedad_meses",       "num"),
    "P05H01A":     ("horas_semanales",        "num"),
    "P03A03A":     ("nivel_educativo",        "cat"),
    "P05C16":      ("categoria_ocupacional",  "cat"),
    "DOMINIO":     ("dominio",                "cat"),
    "OCUPADOS":    ("ocupado",                "int"),
    "NUM_HOGAR":   ("NUM_HOGAR",              "id"),
    "NUM_PERSONA": ("NUM_PERSONA",            "id"),
    "FACTOR":      ("FACTOR",                 "num"),
    "ANIO":        ("ANIO",                   "int"),
    "TRIMESTRE":   ("TRIMESTRE",              "int"),
}
COLS_ORIGINALES = list(COLUMNAS)

TIPO_SPARK = {"num": T.DoubleType(), "cat": T.StringType(),
              "int": T.IntegerType(), "id": T.LongType()}

# Esquema de lectura: todo como texto, para no dejar que pandas/Spark infieran tipos distintos por archivo
ESQUEMA_CRUDO = T.StructType([T.StructField(c, T.StringType(), True) for c in COLS_ORIGINALES])

# Esquema final explícito y en orden fijo (igual para todos los archivos)
ESQUEMA_FINAL = T.StructType(
    [T.StructField(nombre, TIPO_SPARK[clase], True) for nombre, clase in COLUMNAS.values()] + [
        T.StructField("periodo_archivo",      T.StringType(),  False),
        T.StructField("anio_archivo",         T.IntegerType(), False),
        T.StructField("trimestre_calendario", T.IntegerType(), False),
        T.StructField("archivo_origen",       T.StringType(),  False),
    ]
)
ORDEN_FINAL = ESQUEMA_FINAL.fieldNames()
print(ESQUEMA_FINAL.simpleString())

## 1. Carga, armonización y calidad de datos

### 1.1 Funciones de lectura y homologación

Estrategia, siguiendo la recomendación del enunciado:

1. Leer cada Excel por separado con pandas/openpyxl, cargando solo las 14 columnas necesarias (`usecols`) y todo como texto (`dtype=str`). Así se controla la memoria y se evita que pandas infiera `int` en un archivo y `float` u `object` en otro.
2. Pasar a Spark con un esquema explícito de texto y hacer ahí la homologación: recorte de espacios, marcas vacías (`""`, `"nan"`, `"NULL"`, …) a nulo y normalización de códigos para que `"01"`, `1`, `"1.0"` y `" 1 "` queden como el mismo código `"1"`.
3. Convertir cada columna a su tipo final y agregar las columnas de período y trazabilidad.
4. Guardar en Parquet (`staging/<periodo>`) y liberar la memoria de pandas antes de pasar al siguiente archivo.

Los encabezados se normalizan a mayúsculas sin espacios para que, si un archivo trae `Dominio` o `p05d01`, se reconozca igual.

In [ ]:
MARCAS_VACIO = ["", "NAN", "NONE", "NULL", "NA", "N/A", "."]

def texto_limpio(col: str):
    # Recorta espacios y convierte marcas de vacío en nulo.
    s = F.trim(F.col(col))
    return F.when(s.isNull() | F.upper(s).isin(*MARCAS_VACIO), F.lit(None)).otherwise(s)

def codigo_normalizado(col: str):
    # "01", "1", "1.0", " 1 " -> "1". Valores no numéricos se conservan en mayúsculas para validarlos luego.
    s = texto_limpio(col)
    d = s.cast("double")
    entero_valido = d.isNotNull() & ~F.isnan(d) & (F.abs(d) < 1e15) & (d == F.floor(d))
    return (F.when(s.isNull(), F.lit(None))
             .when(entero_valido, d.cast("long").cast("string"))
             .otherwise(F.upper(s)))

def convertir(col: str, clase: str):
    if clase == "num":
        return texto_limpio(col).cast("double")
    if clase == "cat":
        return codigo_normalizado(col)
    if clase == "int":
        return codigo_normalizado(col).cast("int")
    if clase == "id":
        return codigo_normalizado(col).cast("long")
    raise ValueError(clase)


def leer_excel(ruta: Path):
    # Devuelve (encabezado completo del archivo, DataFrame de pandas con las columnas requeridas como texto).
    encabezado = [str(c) for c in pd.read_excel(ruta, nrows=0, engine="openpyxl").columns]
    normal = {c: c.strip().upper() for c in encabezado}
    faltantes = set(COLS_ORIGINALES) - set(normal.values())
    if faltantes:
        raise ValueError(f"{ruta.name}: no contiene las columnas {sorted(faltantes)}")

    usar = [c for c in encabezado if normal[c] in COLUMNAS]
    pdf = pd.read_excel(ruta, usecols=usar, dtype=str, engine="openpyxl")
    pdf = pdf.rename(columns=normal)[COLS_ORIGINALES]
    pdf = pdf.astype(object).where(pdf.notna(), None)
    return [normal[c] for c in encabezado], pdf


def armonizar(sdf_crudo: DataFrame, meta) -> DataFrame:
    # Renombra, tipa y agrega las columnas de período / trazabilidad.
    cols = [convertir(orig, clase).alias(nombre) for orig, (nombre, clase) in COLUMNAS.items()]
    cols += [
        F.lit(meta.periodo_archivo).alias("periodo_archivo"),
        F.lit(int(meta.anio_archivo)).cast("int").alias("anio_archivo"),
        F.lit(int(meta.trimestre_calendario)).cast("int").alias("trimestre_calendario"),
        F.lit(meta.archivo).alias("archivo_origen"),
    ]
    return sdf_crudo.select(*cols).select(ORDEN_FINAL)


def contar_no_convertibles(sdf_crudo: DataFrame) -> dict:
    # Valores con contenido en el Excel que no pudieron convertirse a su tipo numérico destino.
    exprs = []
    for orig, (nombre, clase) in COLUMNAS.items():
        if clase == "cat":
            continue
        tiene_texto = texto_limpio(orig).isNotNull()
        convertido = convertir(orig, clase)
        exprs.append(F.sum((tiene_texto & convertido.isNull()).cast("int")).alias(nombre))
    return sdf_crudo.agg(*exprs).first().asDict()

### 1.2 Conversión de cada archivo a Parquet

Cada archivo se procesa de forma independiente: lectura a Spark con esquema crudo, armonización, Parquet y liberación de memoria. El archivo de 2026 se convierte con exactamente las mismas funciones, pero no se une con 2025.

In [ ]:
registro_carga, encabezados, no_convertibles = [], {}, []

for meta in ARCHIVOS.itertuples(index=False):
    ruta = RAW_DIR / meta.archivo
    print(f"Procesando {meta.periodo_archivo} <- {ruta.name} ...", end=" ")

    encabezado, pdf = leer_excel(ruta)
    encabezados[meta.periodo_archivo] = encabezado

    sdf_crudo = spark.createDataFrame(pdf, schema=ESQUEMA_CRUDO)
    no_convertibles.append({"periodo_archivo": meta.periodo_archivo, **contar_no_convertibles(sdf_crudo)})

    sdf = armonizar(sdf_crudo, meta)
    destino = STAGING_DIR / meta.periodo_archivo
    sdf.write.mode("overwrite").parquet(str(destino))

    registro_carga.append({
        "periodo_archivo": meta.periodo_archivo,
        "archivo_origen": meta.archivo,
        "columnas_excel": len(encabezado),
        "filas_excel": len(pdf),
    })
    print(f"{len(pdf):,} filas, {len(encabezado)} columnas")

    del pdf, sdf_crudo, sdf
    gc.collect()

### 1.3 Lectura desde Parquet y unión de 2025 con `unionByName`

`unionByName` empareja las columnas por nombre, no por posición. Se usa `allowMissingColumns=False` a propósito: si algún archivo no tuviera exactamente las mismas columnas, la unión falla en lugar de rellenar con nulos en silencio.

In [ ]:
staging = {p: spark.read.parquet(str(STAGING_DIR / p)) for p in ARCHIVOS["periodo_archivo"]}

# Todos los archivos deben terminar con el mismo esquema explícito
for p, sdf in staging.items():
    assert sdf.schema.simpleString() == staging["2025T1"].schema.simpleString(), f"Esquema distinto en {p}"

PERIODOS_2025 = ARCHIVOS.loc[ARCHIVOS["anio_archivo"] == 2025, "periodo_archivo"].tolist()

df_2025_raw = reduce(
    lambda a, b: a.unionByName(b, allowMissingColumns=False),
    [staging[p] for p in PERIODOS_2025],
).persist()

df_2026_raw = staging["2026T1"].persist()

print(f"2025 (4 archivos unidos): {df_2025_raw.count():,} filas")
print(f"2026 (prueba, separado): {df_2026_raw.count():,} filas")

### 1.4 Esquema y cinco registros de las columnas seleccionadas

In [ ]:
df_2025_raw.printSchema()

In [ ]:
df_2025_raw.show(5, truncate=False)

### 1.5 Número de registros por archivo antes de los filtros

Se compara lo leído con los conteos publicados en el enunciado. Los conteos después de filtros se agregan en la etapa de filtrado.

In [ ]:
conteo_spark = (
    reduce(DataFrame.unionByName, staging.values())
    .groupBy("periodo_archivo").count()
    .withColumnRenamed("count", "filas_parquet")
    .toPandas()
)

tabla_registros = (
    ARCHIVOS[["periodo_archivo", "uso", "registros_esperados", "columnas_esperadas"]]
    .merge(pd.DataFrame(registro_carga), on="periodo_archivo")
    .merge(conteo_spark, on="periodo_archivo")
)
tabla_registros["filas_ok"]    = tabla_registros["filas_parquet"] == tabla_registros["registros_esperados"]
tabla_registros["columnas_ok"] = tabla_registros["columnas_excel"] == tabla_registros["columnas_esperadas"]
display(tabla_registros)

### 1.6 ¿Por qué IV de 2025 no puede apilarse por posición?

Se compara la posición (índice de columna, base 0) que ocupa cada variable requerida en el encabezado de cada archivo, y qué columnas tiene IV de 2025 que no aparecen en los demás.

In [ ]:
posiciones = pd.DataFrame({
    p: {c: enc.index(c) for c in COLS_ORIGINALES}
    for p, enc in encabezados.items()
})
posiciones["misma_posicion_en_todos"] = posiciones.nunique(axis=1) == 1
display(posiciones)

ref = set(encabezados["2025T1"])
extra_t4 = [c for c in encabezados["2025T4"] if c not in ref]
falta_t4 = [c for c in encabezados["2025T1"] if c not in set(encabezados["2025T4"])]
mismo_orden_t1_t3 = encabezados["2025T1"] == encabezados["2025T2"] == encabezados["2025T3"]

print(f"Columnas en 2025T4 que no están en 2025T1: {len(extra_t4)}")
print("  ", extra_t4[:40], "..." if len(extra_t4) > 40 else "")
print(f"Columnas de 2025T1 ausentes en 2025T4: {len(falta_t4)}")
print(f"¿T1, T2 y T3 tienen exactamente el mismo encabezado y orden? {mismo_orden_t1_t3}")

**Respuesta.** IV de 2025 trae 302 columnas frente a 270 de los demás archivos. Las columnas adicionales (listadas arriba) desplazan la posición de las variables que se ubican después de ellas, como muestra la tabla de posiciones. Un apilado posicional (`union`, o concatenar por índice) pegaría el contenido de la columna *k* de IV de 2025 debajo de la columna *k* de los otros archivos, aunque sean preguntas distintas. El resultado mezclaría variables (por ejemplo, un código de otra pregunta quedaría como salario u horas) sin producir ningún error, porque solo se exige que coincida el número de columnas o que los tipos sean compatibles.

Por eso se seleccionan las columnas por nombre en cada archivo, se llevan todas al mismo esquema explícito y se unen con `unionByName`, que empareja por nombre y falla si falta alguna columna.

### 1.7 Auditoría de `TRIMESTRE` y `ANIO` frente al archivo de procedencia

Se conserva `TRIMESTRE` original, pero no se usa como trimestre calendario. La tabla cruza el archivo de origen con los valores observados.

In [ ]:
auditoria_trimestre = (
    reduce(DataFrame.unionByName, staging.values())
    .groupBy("periodo_archivo", "anio_archivo", "trimestre_calendario", "ANIO", "TRIMESTRE")
    .count()
    .orderBy("periodo_archivo", "TRIMESTRE")
    .toPandas()
)
display(auditoria_trimestre)

En II de 2025 aparecen registros con `TRIMESTRE = 2` junto con la mayoría en `3`. Restar uno a `TRIMESTRE` asignaría esos registros al trimestre calendario 1 aunque provienen del archivo publicado de II de 2025. Por eso `periodo_archivo`, `anio_archivo` y `trimestre_calendario` se derivan del archivo (catálogo de la sección 0.1), y `archivo_origen` permite rastrear cada fila hasta su fuente. Esta asignación no corrige ni reinterpreta la respuesta original, que queda intacta en `TRIMESTRE`.

### 1.8 Valores no convertibles

Celdas que tenían contenido en el Excel pero no pudieron convertirse al tipo numérico destino (por ejemplo, texto en una columna de montos). Después de la conversión quedan como nulos, así que conviene cuantificarlas por separado de los vacíos originales.

In [ ]:
tabla_no_convertibles = pd.DataFrame(no_convertibles).set_index("periodo_archivo")
display(tabla_no_convertibles)
print("Total no convertibles:", int(tabla_no_convertibles.to_numpy().sum()))

### 1.9 Faltantes por variable antes de aplicar filtros

Se considera faltante un valor nulo (vacío en el Excel o no convertible) y, en las columnas `double`, también `NaN`. El cálculo se hace sobre todas las filas de 2025, antes de cualquier filtro.

In [ ]:
COLS_ANALITICAS = [nombre for nombre, _ in COLUMNAS.values()]
TIPOS = dict(df_2025_raw.dtypes)

def es_faltante(c):
    cond = F.col(c).isNull()
    if TIPOS[c] == "double":
        cond = cond | F.isnan(F.col(c))
    return cond

def tabla_faltantes(df: DataFrame, por=None) -> pd.DataFrame:
    exprs = [F.count(F.lit(1)).alias("_n")] + [
        F.sum(es_faltante(c).cast("int")).alias(c) for c in COLS_ANALITICAS
    ]
    return (df.groupBy(por) if por else df).agg(*exprs).toPandas()

tot = tabla_faltantes(df_2025_raw).iloc[0]
n_2025 = int(tot["_n"])
faltantes_2025 = pd.DataFrame({
    "faltantes": [int(tot[c]) for c in COLS_ANALITICAS],
}, index=COLS_ANALITICAS)
faltantes_2025["porcentaje"] = (100 * faltantes_2025["faltantes"] / n_2025).round(2)
faltantes_2025 = faltantes_2025.sort_values("porcentaje", ascending=False)
print(f"Registros 2025 antes de filtros: {n_2025:,}")
display(faltantes_2025)

In [ ]:
por_periodo = tabla_faltantes(
    reduce(DataFrame.unionByName, staging.values()), por="periodo_archivo"
).set_index("periodo_archivo").sort_index()
pct_por_periodo = (100 * por_periodo[COLS_ANALITICAS].div(por_periodo["_n"], axis=0)).round(2)
print("% de faltantes por archivo (incluye 2026 como referencia):")
display(pct_por_periodo.T)

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
datos = faltantes_2025.sort_values("porcentaje")
ax.barh(datos.index, datos["porcentaje"], color="#2b8cbe")
for i, v in enumerate(datos["porcentaje"]):
    ax.text(v + 0.5, i, f"{v:.1f}%", va="center", fontsize=9)
ax.set_xlabel("% de registros con valor faltante")
ax.set_title(f"Faltantes por variable antes de filtros - 2025 (n = {n_2025:,})")
ax.set_xlim(0, max(100, datos["porcentaje"].max() + 8))
plt.tight_layout()
plt.savefig(FIG_DIR / "01_faltantes_2025.png", dpi=120)
plt.show()

### 1.10 ¿Faltante porque la pregunta no aplica, o respuesta no registrada?

Las variables laborales (salario, antigüedad, horas, categoría) solo se preguntan a quienes están ocupados, y el salario `P05D01` solo a quienes son asalariados. Si se separa la muestra según el flujo del cuestionario, se ve de dónde vienen los faltantes.

In [ ]:
flujo = (
    F.when(F.col("ocupado") != 1, "1. No ocupado / sin dato de ocupación")
     .when(F.col("ocupado").isNull(), "1. No ocupado / sin dato de ocupación")
     .when(F.col("categoria_ocupacional").isin("1", "2", "3", "4"), "3. Ocupado asalariado (P05C16 1-4)")
     .otherwise("2. Ocupado no asalariado")
)
VARS_LABORALES = ["salario_mensual", "antiguedad_anios", "antiguedad_meses", "horas_semanales"]

faltantes_flujo = (
    df_2025_raw.withColumn("grupo_flujo", flujo)
    .groupBy("grupo_flujo")
    .agg(F.count(F.lit(1)).alias("n"),
         *[F.round(100 * F.avg(es_faltante(c).cast("int")), 2).alias(f"%falta_{c}") for c in VARS_LABORALES])
    .orderBy("grupo_flujo")
    .toPandas()
)
display(faltantes_flujo)

**Respuesta.** Un dato puede faltar por dos razones que significan cosas distintas:

- La pregunta no corresponde (salto de flujo). El cuestionario no le hace la pregunta a esa persona porque no aplica. A alguien no ocupado no se le pregunta su salario ni sus horas, y a un trabajador por cuenta propia no se le pregunta `P05D01`. El vacío es *estructural* y esperado: no hay un valor real desconocido, simplemente la variable no está definida para esa persona. Estos registros no se imputan; se excluyen porque están fuera de la población de análisis (asalariados ocupados).
- Respuesta no registrada (no respuesta). La pregunta sí aplicaba, pero no hay valor: la persona no sabía o no quiso responder, o hubo un error de captura. Aquí el valor existe pero se desconoce. Es un problema de calidad que puede sesgar resultados si la no respuesta se relaciona con el propio salario (por ejemplo, si quienes ganan más responden menos).

La tabla de la celda anterior permite distinguirlos en la práctica. Los faltantes de salario entre no ocupados y ocupados no asalariados son saltos de flujo. Los faltantes que quedan dentro del grupo *ocupado asalariado* son los que realmente corresponden a respuestas no registradas, y son los que se deben reportar como pérdida de información de la población analítica.

En nuestros datos, el salario falta en ___ % de los asalariados ocupados, frente a ___ % en el total de 2025.

---
## 3. Relaciones entre variables numéricas

Las correlaciones se calculan sobre todos los registros elegibles de 2025 (conjunto `prep_2025`, ya filtrado en el Ejercicio 1), sin muestrear. La muestra de hasta 5,000 filas se usa solo para el diagrama de dispersión.

Variables: `salario_mensual`, `edad`, `antiguedad` (años, construida como `antiguedad_anios + antiguedad_meses / 12`) y `horas_semanales`.

In [ ]:
assert PREP_2025_DIR.exists(), (
    f"No existe {PREP_2025_DIR}. Primero hay que correr los filtros del Ejercicio 1 y guardar el parquet."
)
df_2025 = spark.read.parquet(str(PREP_2025_DIR)).persist()

VARS_CORR = ["salario_mensual", "edad", "antiguedad", "horas_semanales"]
ETIQUETAS = {"salario_mensual": "Salario mensual", "edad": "Edad",
             "antiguedad": "Antigüedad (años)", "horas_semanales": "Horas semanales"}

faltan_cols = set(VARS_CORR) - set(df_2025.columns)
assert not faltan_cols, f"prep_2025 no tiene {faltan_cols}"

n_elegibles = df_2025.count()
print(f"Registros elegibles 2025: {n_elegibles:,}")

In [ ]:
ensamblador = VectorAssembler(inputCols=VARS_CORR, outputCol="features_corr", handleInvalid="skip")
vec_2025 = ensamblador.transform(df_2025.select(VARS_CORR)).select("features_corr")

n_vector = vec_2025.count()
print(f"Filas usadas en la correlación: {n_vector:,} (descartadas por nulos/NaN: {n_elegibles - n_vector:,})")

def matriz_correlacion(metodo: str) -> pd.DataFrame:
    m = Correlation.corr(vec_2025, "features_corr", metodo).head()[0].toArray()
    nombres = [ETIQUETAS[c] for c in VARS_CORR]
    return pd.DataFrame(m, index=nombres, columns=nombres)

corr_pearson = matriz_correlacion("pearson")
print("Matriz de correlación de Pearson (2025, todos los registros elegibles):")
display(corr_pearson.round(3))

In [ ]:
fig, ax = plt.subplots(figsize=(6.5, 5.5))
mascara = np.triu(np.ones_like(corr_pearson, dtype=bool), k=1)
sns.heatmap(corr_pearson, mask=mascara, annot=True, fmt=".3f", cmap="RdBu_r",
            vmin=-1, vmax=1, center=0, square=True, linewidths=0.5,
            cbar_kws={"label": "r de Pearson"}, ax=ax)
ax.set_title(f"Correlación de Pearson - 2025 (n = {n_vector:,})")
plt.tight_layout()
plt.savefig(FIG_DIR / "03_heatmap_pearson.png", dpi=120)
plt.show()

### 3.1 Complemento: correlación de Spearman

El salario suele ser muy asimétrico a la derecha y la correlación de Pearson es sensible a valores extremos. La correlación de Spearman (basada en rangos, también con `Correlation.corr()`) mide asociación monótona, no necesariamente lineal. Si ambas difieren mucho, parte de la asociación de Pearson está siendo dominada por pocos salarios altos o la relación no es lineal. Es un complemento; la matriz solicitada es la de Pearson.

In [ ]:
corr_spearman = matriz_correlacion("spearman")

comparacion = pd.DataFrame({
    "Pearson":  corr_pearson["Salario mensual"].drop("Salario mensual"),
    "Spearman": corr_spearman["Salario mensual"].drop("Salario mensual"),
})
comparacion["|Pearson|"] = comparacion["Pearson"].abs()
comparacion = comparacion.sort_values("|Pearson|", ascending=False).drop(columns="|Pearson|")
print("Asociación con el salario mensual (ordenado por |r de Pearson|):")
display(comparacion.round(3))

r_edad_antig = corr_pearson.loc["Edad", "Antigüedad (años)"]
print(f"r(Edad, Antigüedad) Pearson = {r_edad_antig:.3f} | "
      f"Spearman = {corr_spearman.loc['Edad', 'Antigüedad (años)']:.3f}")

In [ ]:
muestra = (
    df_2025.select(VARS_CORR)
    .sample(fraction=min(1.0, 1.3 * MAX_FILAS_GRAFICO / n_elegibles), seed=SEED)
    .limit(MAX_FILAS_GRAFICO)
    .toPandas()
)

fig, axes = plt.subplots(1, 3, figsize=(16, 4.8))

axes[0].scatter(muestra["edad"], muestra["antiguedad"], s=6, alpha=0.3)
lim = [15, muestra["edad"].max()]
axes[0].plot(lim, [l - 15 for l in lim], "r--", lw=1, label="antigüedad = edad − 15")
axes[0].set_xlabel("Edad (años)"); axes[0].set_ylabel("Antigüedad (años)")
axes[0].set_title("Edad vs antigüedad"); axes[0].legend()

axes[1].scatter(muestra["edad"], muestra["salario_mensual"], s=6, alpha=0.3)
axes[1].set_yscale("log")
axes[1].set_xlabel("Edad (años)"); axes[1].set_ylabel("Salario mensual (Q, escala log)")
axes[1].set_title("Edad vs salario")

axes[2].scatter(muestra["horas_semanales"], muestra["salario_mensual"], s=6, alpha=0.3)
axes[2].set_yscale("log")
axes[2].set_xlabel("Horas habituales por semana"); axes[2].set_ylabel("Salario mensual (Q, escala log)")
axes[2].set_title("Horas vs salario")

fig.suptitle(f"Muestra aleatoria de {len(muestra):,} registros (solo para visualizar; "
             f"las correlaciones usan los {n_vector:,})", y=1.02)
plt.tight_layout()
plt.savefig(FIG_DIR / "03_dispersion.png", dpi=120, bbox_inches="tight")
plt.show()

### 3.2 Interpretación

**¿Qué variables presentan mayor asociación lineal con el salario?**

La variable con mayor |r| respecto al salario es [variable] (r = [valor]), seguida de [variable] (r = [valor]) y [variable] (r = [valor]).

Como referencia para interpretar la magnitud: |r| < 0.1 es prácticamente nula, 0.1–0.3 débil, 0.3–0.5 moderada y > 0.5 fuerte. En datos de salarios es habitual que las correlaciones con variables individuales sean débiles o moderadas: el salario depende en buena medida de factores categóricos (educación, sector público/privado, dominio urbano/rural) que no entran en esta matriz y que se modelan en los Ejercicios 5 y 6.

Al comparar con Spearman:
- Si Spearman > Pearson para una variable, la relación es monótona pero no lineal, o los salarios extremos debilitan la correlación lineal.
- Si Pearson > Spearman, pocos salarios muy altos pueden estar inflando la asociación lineal.

Las correlaciones describen asociación en los registros analizados, no efectos causales: un r positivo con la antigüedad no implica que acumular antigüedad cause un salario mayor.

**¿Existe relación entre edad y antigüedad?**

r(Edad, Antigüedad) = [valor]. Se espera una relación positiva, y el diagrama de dispersión muestra por qué no es perfecta. La antigüedad está acotada por la edad: nadie puede llevar en su empleo actual más años de los que ha vivido (filtro `antigüedad ≤ edad`), y en la práctica casi nadie más años que los transcurridos desde la edad mínima para trabajar. Por eso los puntos forman un triángulo con la mayoría bajo la línea de referencia `antigüedad = edad − 15`; los que la superan corresponden a personas que empezaron en su empleo actual antes de los 15 años. Una persona mayor *puede* tener mucha antigüedad, pero también puede haber cambiado de empleo recientemente, lo que produce mucha dispersión en edades altas: la edad fija un techo, no un valor.

Implicaciones para el modelado: si la correlación es alta, edad y antigüedad aportan información parcialmente redundante (colinealidad). En la regresión lineal esto vuelve inestables sus coeficientes individuales y hay que interpretarlos con cuidado, aunque no afecta mucho la capacidad predictiva. En Random Forest es menos problemático, aunque la importancia se reparte entre ambas. En KMeans, dos variables muy correlacionadas pesan más en la distancia que una sola.

---
### Liberación de memoria

In [ ]:
for d in (df_2025_raw, df_2026_raw, df_2025):
    d.unpersist()